# **FINITE ELEMENTS in TIME in IRKSOME**

---

## **Install**

### Firedrake

In [14]:
try:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh"
    !bash "/tmp/firedrake-install.sh"
    from firedrake import *  # noqa: F401
except:
    from firedrake import *  # noqa: F401

--2026-06-03 00:53:28--  https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh
Resolving fem-on-colab.github.io (fem-on-colab.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to fem-on-colab.github.io (fem-on-colab.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4767 (4.7K) [application/x-sh]
Saving to: ‘/tmp/firedrake-install.sh’

/tmp/firedrake-inst 100%[===================>]   4.66K  --.-KB/s    in 0s      

2026-06-03 00:53:28 (29.2 MB/s) - ‘/tmp/firedrake-install.sh’ saved [4767/4767]

+ INSTALL_PREFIX=/usr/local
++ echo /usr/local
++ awk -F/ '{print NF-1}'
+ INSTALL_PREFIX_DEPTH=2
+ PROJECT_NAME=fem-on-colab
+ SHARE_PREFIX=/usr/local/share/fem-on-colab
+ FIREDRAKE_INSTALLED=/usr/local/share/fem-on-colab/firedrake.installed
+ [[ ! -f /usr/local/share/fem-on-colab/firedrake.installed ]]
+ set +x
























#############################################################

### Irksome

In [15]:
try:
    !python3 -m pip install --no-dependencies git+https://github.com/firedrakeproject/Irksome.git
    from irksome import *  # noqa: F401
except:
    from irksome import *  # noqa: F401

  Cloning https://github.com/firedrakeproject/Irksome.git to /tmp/pip-req-build-5cf_ugd5
  Running command git clone --filter=blob:none --quiet https://github.com/firedrakeproject/Irksome.git /tmp/pip-req-build-5cf_ugd5
  Resolved https://github.com/firedrakeproject/Irksome.git to commit 010d31ea0cbc091cf163a8558ef940afe89fd1e3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### Other

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

# Paper-style colours (matching the manuscript figures)
seabornblue   = "#4C72B0"
seaborngreen  = "#55A868"
seabornred    = "#C44E52"
seabornpurple = "#8172B2"
seabornorange = "#F4953B"

---

## **Functions**

### $N$-body problem

In [19]:
# ============================================================================
# State space: N particles in 2D, positions q and momenta p. Irksome wants
# Firedrake `Function`s, so we use a trivial mesh and the "Real" element
# (globally constant DoFs) — the Function holds a 2N-vector with no spatial
# dependence whatsoever.
# ============================================================================
N = 5  # number of particles

G    = Constant(1.0)
mass = Constant(1.0)
eps2 = Constant(0.05**2)  # softening, squared

mesh = UnitIntervalMesh(1)
Vq = VectorFunctionSpace(mesh, "R", 0, dim=2*N)
Vp = VectorFunctionSpace(mesh, "R", 0, dim=2*N)
Z = Vq * Vp

state = Function(Z)
q, p = split(state)
vq, vp = TestFunctions(Z)

def vec(field, i):
    """Extract the i-th particle's 2D sub-vector from a flat 2N-vector."""
    return as_vector([field[2*i], field[2*i+1]])

# ============================================================================
# Hamiltonian as a UFL expression — used for diagnostics.
# ============================================================================
def kinetic(p_field):
    return sum(inner(vec(p_field, i), vec(p_field, i)) / (2 * mass)
               for i in range(N))

def potential(q_field):
    return - sum(G * mass**2
                 / sqrt(inner(vec(q_field, i) - vec(q_field, j),
                              vec(q_field, i) - vec(q_field, j)) + eps2)
                 for i in range(N) for j in range(i+1, N))

H_form = (kinetic(p) + potential(q)) * dx

# ============================================================================
# Variational form encoding Hamilton's equations.
#   dq/dt - p/m         = 0,  tested against vq
#   dp/dt + dU/dq       = 0,  tested against vp
# ============================================================================
F_integrand = inner(Dt(q) - p / mass, vq) + inner(Dt(p), vp)
for i in range(N):
    qi, vpi = vec(q, i), vec(vp, i)
    for j in range(N):
        if j == i:
            continue
        diff = qi - vec(q, j)
        r2   = inner(diff, diff) + eps2
        F_integrand += G * mass**2 * inner(diff, vpi) / r2**1.5
F = F_integrand * dx

# ============================================================================
# Initial conditions: positions and momenta drawn from a Gaussian, with
# momenta shifted so the total system momentum vanishes (centre of mass
# stays put).
# ============================================================================
sigma_q = 1.0    # std dev of initial positions
sigma_p = 0.1    # std dev of initial momenta

def set_initial_conditions(seed=0):
    rng = np.random.default_rng(seed)
    q0 = rng.standard_normal(2*N) * sigma_q
    p0 = rng.standard_normal(2*N) * sigma_p
    p0 = p0.reshape(N, 2)
    p0 -= p0.mean(axis=0)
    p0 = p0.reshape(-1)
    state.sub(0).dat.data[:] = q0
    state.sub(1).dat.data[:] = p0

# ============================================================================
# Time-stepping driver — takes a scheme name (looked up in the `schemes`
# dict defined below the fold) and a polynomial degree, returns a dict of
# time series for plotting.
# ============================================================================
solver_params = {
    "snes_type": "newtonls",
    "snes_rtol": 1e-12,
    "snes_atol": 1e-12,
    "ksp_type": "preonly",
    "pc_type": "lu",
}

def run(scheme_name, deg=2, T=50.0, Nt=2000, seed=0):
    set_initial_conditions(seed)
    scheme  = schemes[scheme_name](deg)
    dt      = Constant(T / Nt)
    t       = Constant(0.0)
    stepper = TimeStepper(F, scheme, t, dt, state,
                          solver_parameters=solver_params)

    times     = np.zeros(Nt + 1)
    positions = np.zeros((Nt + 1, 2*N))
    energies  = np.zeros(Nt + 1)
    positions[0] = state.sub(0).dat.data_ro.copy()
    energies[0]  = float(assemble(H_form))

    for k in range(Nt):
        stepper.advance()
        t.assign(float(t) + float(dt))
        times[k+1]     = float(t)
        positions[k+1] = state.sub(0).dat.data_ro.copy()
        energies[k+1]  = float(assemble(H_form))

    return {"times": times, "positions": positions, "energies": energies}

# ============================================================================
# Plotting helpers.
# ============================================================================
_palette = {"cpg(2)": seabornblue, "gl(1)": seabornred, "gl(2)": seabornorange}

def plot_energy_drift(results):
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, res in results.items():
        rel = (res["energies"] - res["energies"][0]) / abs(res["energies"][0])
        ax.plot(res["times"], rel, label=name,
                color=_palette.get(name), lw=1.5)
    ax.set_xlabel("$t$")
    ax.set_ylabel(r"$(H(t) - H(0)) \,/\, |H(0)|$")
    ax.set_title("Relative Hamiltonian drift")
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.legend()
    fig.tight_layout()
    plt.show()

def plot_orbits_grid(results):
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1:
        axes = [axes]
    cmap = plt.get_cmap("viridis")
    for ax, (name, res) in zip(axes, results.items()):
        positions = res["positions"]
        times     = res["times"]
        norm = plt.Normalize(times.min(), times.max())
        for i in range(N):
            x = positions[:, 2*i]
            y = positions[:, 2*i+1]
            pts  = np.array([x, y]).T.reshape(-1, 1, 2)
            segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
            lc = LineCollection(segs, cmap=cmap, norm=norm,
                                lw=0.5, alpha=0.85)
            lc.set_array(times[:-1])
            ax.add_collection(lc)
        all_x = positions[:, 0::2]
        all_y = positions[:, 1::2]
        pad = 0.1 * max(all_x.ptp(), all_y.ptp())
        ax.set_xlim(all_x.min() - pad, all_x.max() + pad)
        ax.set_ylim(all_y.min() - pad, all_y.max() + pad)
        ax.set_aspect("equal")
        ax.set_title(name)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.tight_layout()
    plt.show()

---

## **Finite elements in time for ODEs**

To introduce the concept of variational-in-time methods, we're going to start by keeping it simple with ODEs only.

### General framework

Take an ODE for $x : [0, T] \to \mathbb{R}^n$:
> $$
> \dot{x} = f(x).
> $$

Suppose we have data $x_n \approx x(t_n)$ and want to advance to $t_{n+1} = t_n + \Delta t$.

A very general framework for an *$s$-stage* time discretisation is: seek a polynomial $x_h$ of degree $s$ on $[t_n, t_{n+1}]$ such that
1. $x_h(t_n) = x_n$ (interpolate the initial data); and
2. $s$ further constraints aim to capture the ODE $\dot{x} = f(x)$.

How do we pick those $s$ further constraints?

### Option 1: Simple collocation $\to$ RK

Require that $\dot{x}_h = f(x_h)$ holds *exactly* at $s$ chosen *collocation points* $\xi_1, \ldots, \xi_s \in [t_n, t_{n+1}]$:
> $$
> \dot{x}_h(\xi_k) = f(x_h(\xi_k)), \qquad k = 1, \ldots, s.
> $$

This recovers **collocation Runge–Kutta (RK) methods**:
- $s = 1$, midpoint node $\implies$ implicit midpoint
- $s$ Gauss–Legendre nodes $\implies$ Gauss–Legendre RK
- $s$ right-Radau nodes $\implies$ Radau IIA

### Option 2: Test against polynomial test functions $\to$ CPG

Let's put on our FEM hats.
Tather than enforcing the ODE *pointwise* at $s$ nodes, we could enforce it *weakly* against polynomial test functions $y_h \in \mathcal{P}_{s-1}(t_n, t_{n+1})$:
> $$
> \int_{t_n}^{t_{n+1}} \dot{x}_h \,y_h\, \mathrm{d}t = \int_{t_n}^{t_{n+1}} f(x_h) \,y_h\, \mathrm{d}t, \qquad \forall y_h \in \mathcal{P}_{s-1}.
> $$

The mismatch in polynomial degrees (solution in $\mathcal{P}_s$, tests in $\mathcal{P}_{s-1}$) is by design. One of the solution's $s+1$ DoFs has already been spent on the initial condition, leaving $s$ unknowns to match the $s$-dimensional test space.

This defines the **continuous Petrov–Galerkin** (CPG) method:
- *"Continuous"*, because the solution function is continuous across timesteps (initial data is strongly enforced).
- *"Petrov–Galerkin"*, because the trial and test spaces differ in polynomial degree.

*(There's also a bit of a sibling FET scheme, discontinuous Galerkin (DG). We won't discuss this here today, but it's also implemented in Irksome.)*

### Why would this ever be useful?

- **CPG generalises collocation RK.**
Approximating the integrals $\int_{t_n}^{t_{n+1}}$ above using an $s$-node quadrature rule, we recover *exactly* the corresponding collocation RK method.
So firstly, we don't lose anything by switching to CPG; we just gain the freedom of choosing other quadrature rules.

- **Structure preservation.**
Secondly, the exact integrals in CPG methods give them some *great* conservative and dissipative properties.
For example, CPG is *energy-conserving* for Hamiltonian systems.
It all boils down to being able to reproduce the fundamental theorem of calculus, $\mathcal{H}(t_{n+1}) - \mathcal{H}(t_n) = \int_{t_n}^{t_{n+1}} \dot{\mathcal{H}}$.
The proof's relatively simple...

### CPG is energy-conserving for Hamiltonian systems

Take a Hamiltonian system in $(\mathbf{q}, \mathbf{p}) \in \mathbb{R}^{2n}$:
> $$
> \dot{\mathbf{q}} = \frac{\partial \mathcal{H}}{\partial \mathbf{p}}(\mathbf{q}, \mathbf{p}), \qquad \dot{\mathbf{p}} = - \frac{\partial \mathcal{H}}{\partial \mathbf{q}}(\mathbf{q}, \mathbf{p}).
> $$

Over $[t_n, t_{n+1}]$, CPG seeks $(\mathbf{q}_h, \mathbf{p}_h) \in \mathcal{P}_s^{2n}$ satisfying initial data $\mathbf{q}_h(t_n) = \mathbf{q}_n$ and $\mathbf{p}_h(t_n) = \mathbf{p}_n$, and such that
> $$
> \int_{t_n}^{t_{n+1}} \dot{\mathbf{q}}_h \cdot \mathbf{r}_h\, \mathrm{d}t = \int_{t_n}^{t_{n+1}} \frac{\partial \mathcal{H}}{\partial \mathbf{p}}(\mathbf{q}_h, \mathbf{p}_h) \cdot \mathbf{r}_h\, \mathrm{d}t,
> \qquad
> \int_{t_n}^{t_{n+1}} \dot{\mathbf{p}}_h \cdot \mathbf{s}_h\, \mathrm{d}t = - \int_{t_n}^{t_{n+1}} \frac{\partial \mathcal{H}}{\partial \mathbf{q}}(\mathbf{q}_h, \mathbf{p}_h) \cdot \mathbf{s}_h\, \mathrm{d}t,
> $$
for all test functions $(\mathbf{r}_h, \mathbf{s}_h) \in \mathcal{P}_{s-1}^{2n}$.

The key observation is that $\dot{\mathbf{q}}_h$ and $\dot{\mathbf{p}}_h$ are themselves degree-$(s-1)$ polynomials, so we are free to test the first equation with $\mathbf{r}_h = \dot{\mathbf{p}}_h$ and the second with $\mathbf{s}_h = \dot{\mathbf{p}}_h$. This gives
> $$
> \int_{t_n}^{t_{n+1}} \dot{\mathbf{q}}_h \cdot \dot{\mathbf{p}}_h\, \mathrm{d}t = \int_{t_n}^{t_{n+1}} \frac{\partial \mathcal{H}}{\partial \mathbf{p}}(\mathbf{q}_h, \mathbf{p}_h) \cdot \dot{\mathbf{p}}_h\, \mathrm{d}t,
> \qquad
> \int_{t_n}^{t_{n+1}} \dot{\mathbf{p}}_h \cdot \dot{\mathbf{q}}_h\, \mathrm{d}t = - \int_{t_n}^{t_{n+1}} \frac{\partial \mathcal{H}}{\partial \mathbf{q}}(\mathbf{q}_h, \mathbf{p}_h) \cdot \dot{\mathbf{q}}_h\, \mathrm{d}t,
> $$
Summing and applying the chain rule,
> $$
> \mathcal{H}(t_{n+1}) - \mathcal{H}(t_n)
> = \int_{t_n}^{t_{n+1}} \dot{\mathcal{H}} \, \mathrm{d}t
> = \int_{t_n}^{t_{n+1}} \!\left( \frac{\partial \mathcal{H}}{\partial \mathbf{q}} \cdot \dot{\mathbf{q}}_h + \frac{\partial \mathcal{H}}{\partial \mathbf{p}} \cdot \dot{\mathbf{p}}_h \right)\! \mathrm{d}t
> = 0,
> $$
So, provided the time integrals are exact, $\mathcal{H}$ is preserved across timesteps regardless of time step size.
Similar results exist for other systems (e.g. gradient descent) and "discrete gradients" show us that any conservation law/dissipation inequality can be preserved discrertely with a well-chosen CPG discretisation.

### $N$-body problem

As a little demo, let's evolve $N$ particles in 2D moving under their own mutual gravity.
This is a Hamiltonian system, with
$$
\mathcal{H} = \sum_{i=1}^{N} \frac{\|\mathbf{p}_i\|^2}{2} - \sum_{i<j} \frac{1}{\sqrt{\|\mathbf{q}_i - \mathbf{q}_j\|^2}}.
$$
We'll set this up with $N = 5$, and compare results from CPG vs Gauss–Legendre RK schemes.

We compare two schemes:
- **CPG(2).** 2-stage CPG (preserves the Hamiltonian)
- **GL(2).** 2-stage Gauss–Legendre (symplectic, but only preserves *quadratic* invariants exactly)

In [ ]:
results = {
    "cpg(2)": run(GalerkinCollocationScheme, deg=2),
    "gl(2)":  run(GaussLegendre,  deg=2),
}

plot_energy_drift(results)
plot_orbits_grid(results)

---

## **TO COME: PDE EXAMPLES**

*To be planned — examples on conservative (Hamiltonian-style) and dissipative (gradient-descent) PDE systems will go here, showing how the same one-line scheme switch carries over from ODEs to PDEs.*